<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.10-agentic-rag/notebooks/exercises-6.10.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.10 — Agentic RAG
Netsetos GenAI Course v2.0 | Module 6

Query routing, Self-RAG/CRAG, LangGraph, LlamaIndex agents, multi-agent, production patterns


In [ ]:
# pip install langgraph langchain langchain-openai llama-index tavily-python -q
print('Ready for Agentic RAG')


## Ex 1: Query Router


In [ ]:
def route_query(query):
    q = query.lower()
    web_kw = ['latest','news','today','current','2026']
    if any(w in q for w in web_kw): return 'web_search'
    knowledge_kw = ['policy','procedure','how to','what is our']
    if any(w in q for w in knowledge_kw): return 'retrieval'
    return 'direct_answer'

for q in ['What is 2+2?','What is our refund policy?','Latest AI news today']:
    print(f'  "{q}" → {route_query(q)}')


## Ex 2: Retrieval Grader


In [ ]:
# from langchain_core.pydantic_v1 import BaseModel, Field
# class GradeDocuments(BaseModel):
#     binary_score: str = Field(description='yes or no')
# grader = llm.with_structured_output(GradeDocuments)
# result = grader.invoke(f'Doc: {doc} Question: {question}')

print('Retrieval grading pipeline:')
print('  1. Retrieve top-k documents')
print('  2. LLM grades each: relevant (yes/no)')
print('  3. Filter out irrelevant docs')
print('  4. If ALL filtered → set web_search=True')
print('  5. Generate with only relevant docs')


## Ex 3: CRAG with Web Fallback


In [ ]:
def crag_evaluate(docs, question):
    '''Simulate CRAG evaluation'''
    scores = [0.8, 0.3, 0.1]  # simulated relevance scores
    upper, lower = 0.7, 0.4
    if any(s > upper for s in scores):
        relevant = [d for d,s in zip(docs,scores) if s > upper]
        return 'correct', relevant
    elif all(s < lower for s in scores):
        return 'incorrect', []  # trigger web search
    else:
        return 'ambiguous', docs  # use both

action, docs = crag_evaluate(['doc1','doc2','doc3'], 'What is RAG?')
print(f'Action: {action}, Docs: {docs}')
print('\nCRAG actions:')
print('  Correct: refine docs (knowledge strips)')
print('  Incorrect: discard all → web search fallback')
print('  Ambiguous: refined docs + web search results')


## Ex 4: LangGraph CRAG Topology


In [ ]:
# from langgraph.graph import StateGraph, START, END
# from typing import TypedDict, List

# class GraphState(TypedDict):
#     question: str
#     documents: List[str]
#     generation: str
#     web_search: str
#     retry_count: int

# workflow = StateGraph(GraphState)
# workflow.add_node('retrieve', retrieve)
# workflow.add_node('grade_documents', grade_documents)
# workflow.add_node('generate', generate)
# workflow.add_node('transform_query', transform_query)
# workflow.add_node('web_search_node', web_search)
# workflow.add_edge(START, 'retrieve')
# workflow.add_conditional_edges('grade_documents', decide_to_generate)
# workflow.add_conditional_edges('generate', grade_generation)

print('LangGraph CRAG topology:')
print('  START → retrieve → grade_documents')
print('    ├─ relevant → generate → hallucination_check')
print('    │                         ├─ grounded → answer_check')
print('    │                         │              ├─ useful → END')
print('    │                         │              └─ not useful → web_search')
print('    │                         └─ not grounded → generate (retry)')
print('    └─ not relevant → transform_query → retrieve (loop)')


## Ex 5: LlamaIndex RouterQueryEngine


In [ ]:
# from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
# from llama_index.core.selectors import LLMSingleSelector
# from llama_index.core.tools import QueryEngineTool

# tools = [
#     QueryEngineTool.from_defaults(query_engine=hr_engine,
#         description='HR policies, benefits, leave'),
#     QueryEngineTool.from_defaults(query_engine=finance_engine,
#         description='Financial reports, budgets, expenses'),
#     QueryEngineTool.from_defaults(query_engine=legal_engine,
#         description='Legal contracts, compliance, regulations'),
# ]
# router = RouterQueryEngine(selector=LLMSingleSelector.from_defaults(),
#     query_engine_tools=tools)

print('RouterQueryEngine: Adaptive RAG at query-engine level')
print('  LLMSingleSelector: LLM chooses tool from descriptions')
print('  PydanticSingleSelector: function-calling for reliable routing')
print('  Tool description quality = #1 factor in routing accuracy')


## Ex 6: Model Tiering + Semantic Caching


In [ ]:
print('Model tiering cost comparison:')
print()
for task, model, cost in [
    ('Grading/routing','GPT-4o-mini','$0.15/MTok'),
    ('Intent classification','Claude Haiku','$0.25/MTok'),
    ('Generation','GPT-4o','$2.50/MTok'),
    ('Complex reasoning','Claude Opus','$15/MTok'),
]: print(f'  {task:25s}: {model:15s} {cost}')
print()
print('Semantic caching:')
print('  Store: query_embedding → response')
print('  Hit: cosine_sim(new_query, cached_query) > 0.95')
print('  Savings: up to 73% for FAQ workloads')
print()
print('Combined savings: 40% (tiering) + 73% cache hits')


## Ex 7: Circuit Breaker with DEGRADED State


In [ ]:
class AgentCircuitBreaker:
    def __init__(self):
        self.state = 'CLOSED'
        self.failure_count = 0
        self.threshold = 5
    
    def record_result(self, success, hallucination=False, loop=False):
        if not success or hallucination or loop:
            self.failure_count += 1
        else:
            self.failure_count = max(0, self.failure_count - 1)
        
        if self.failure_count >= self.threshold:
            self.state = 'OPEN'
        elif self.failure_count >= self.threshold // 2:
            self.state = 'DEGRADED'  # 4th state for agents
        else:
            self.state = 'CLOSED'
        return self.state
    
    def get_fallback(self):
        fallbacks = {
            'CLOSED': 'agentic_rag',
            'DEGRADED': 'simple_rag',
            'OPEN': 'cached_response',
        }
        return fallbacks[self.state]

cb = AgentCircuitBreaker()
for ok in [True,True,False,False,False]:
    state = cb.record_result(ok)
    print(f'  Success={ok} → State={state} → Fallback={cb.get_fallback()}')


## Ex 8: RAGAS Evaluation


In [ ]:
print('RAGAS metrics for agentic RAG:')
print()
for metric, desc in [
    ('ToolCallF1','Correct tool execution accuracy'),
    ('AgentGoalAccuracy','Did agent achieve the user\'s goal?'),
    ('TopicAdherence','Did agent stay within allowed domains?'),
    ('Faithfulness','Is answer grounded in retrieved docs?'),
    ('AnswerRelevancy','Does answer address the question?'),
    ('ContextPrecision','Are retrieved docs relevant?'),
    ('ContextRecall','Did we retrieve all needed info?'),
]: print(f'  {metric:25s}: {desc}')
print()
print('Production gate: faithfulness >= 0.80')
print('Integrate into CI/CD with LangSmith or Langfuse')


---
## Recap
Agentic RAG adds decision loops to standard RAG: retrieve → evaluate → retry → verify. Five research patterns: Self-RAG (5.8% hallucination, requires fine-tuning), CRAG (10.5%, plug-and-play — start here), Adaptive RAG (cost optimization via complexity routing), FLARE (long-form generation), IRCoT (multi-hop reasoning). LangGraph: StateGraph with typed state, 5 nodes, conditional edges, hallucination + answer grading. LlamaIndex: event-driven Workflows, RouterQueryEngine, SubQuestionQueryEngine. Multi-agent: 3× cost for ~5% quality gain — use only when architecturally necessary. Production: model tiering (40% savings), semantic caching (73%), circuit breakers with DEGRADED state, RAGAS faithfulness ≥ 0.80. India: regulatory routing (GST/IT/SEBI/RBI), Sarvam 105B, Vyakyarth embeddings, Krutrim Cloud, IndiaAI Mission compute.
